# Notebook Ingestion Debug

Interactive debug notebook for `utils.ingest.ingest_ipynb_document`.


In [1]:
from pathlib import Path
import json
import base64
import io

from PIL import Image as PILImage

from utils.ingest import ingest_ipynb_document, ingest_document
from utils.summarize import summarize_objects


In [2]:
# Choose notebook to test
NOTEBOOK_PATH = Path("test.ipynb")
TOKENIZER_PATH = "local_tokenizer/embeddinggemma"
SHOW = 3

assert NOTEBOOK_PATH.exists(), f"Notebook not found: {NOTEBOOK_PATH}"
print("Testing notebook:", NOTEBOOK_PATH)


Testing notebook: test.ipynb


In [3]:
# Run direct ingestion
texts, tables, images = ingest_ipynb_document(
    str(NOTEBOOK_PATH),
    tokenizer_model_path=TOKENIZER_PATH,
)

print(
    "ingest_ipynb_document counts ->",
    f"texts={len(texts)}",
    f"tables={len(tables)}",
    f"images={len(images)}",
)


ingest_ipynb_document counts -> texts=17 tables=2 images=1


In [4]:
# Validate dispatcher parity
d_texts, d_tables, d_images = ingest_document(
    str(NOTEBOOK_PATH),
    tokenizer_model_path=TOKENIZER_PATH,
)

print(
    "ingest_document counts ->",
    f"texts={len(d_texts)}",
    f"tables={len(d_tables)}",
    f"images={len(d_images)}",
)


ingest_document counts -> texts=17 tables=2 images=1


In [ ]:
def clip(value, max_len=250):
    value = value or ""
    value = value.replace("", "\n")
    return value if len(value) <= max_len else value[:max_len] + "..."


def safe_json(value):
    try:
        return json.dumps(value, indent=2, default=str)
    except Exception:
        return str(value)


def image_debug_meta(image_b64):
    try:
        raw = base64.b64decode(image_b64)
        with PILImage.open(io.BytesIO(raw)) as img:
            return {"format": img.format, "mode": img.mode, "size": img.size}
    except Exception as exc:
        return {"error": str(exc)}


In [8]:
# Inspect text chunks
for i, obj in enumerate(texts[SHOW:-1]):
    print(f"\n[text {i}]")
    print("metadata:", safe_json(getattr(obj, "metadata", {})))
    print("content:", getattr(obj, "text", ""))



[text 0]
metadata: {
  "filename": "test.ipynb",
  "origin": "ipynb",
  "pages": [],
  "bboxes": [],
  "cell_index": 3,
  "cell_type": "code",
  "output_index": null,
  "mime_type": null,
  "sequence_index": 5,
  "chunk_index": 0
}
content: [code cell 3]
print("Notebook ingestion smoke test")

[text 1]
metadata: {
  "filename": "test.ipynb",
  "origin": "ipynb",
  "pages": [],
  "bboxes": [],
  "cell_index": 3,
  "cell_type": "code",
  "output_index": 0,
  "mime_type": "text/plain",
  "sequence_index": 6,
  "chunk_index": 0
}
content: [output stream]
Notebook ingestion smoke test\n

[text 2]
metadata: {
  "filename": "test.ipynb",
  "origin": "ipynb",
  "pages": [],
  "bboxes": [],
  "cell_index": 4,
  "cell_type": "code",
  "output_index": null,
  "mime_type": null,
  "sequence_index": 7,
  "chunk_index": 0
}
content: [code cell 4]
# Simulated markdown table output

[text 3]
metadata: {
  "filename": "test.ipynb",
  "origin": "ipynb",
  "pages": [],
  "bboxes": [],
  "cell_index": 4,

In [ ]:
# Inspect table objects
for i, obj in enumerate(tables[:SHOW]):
    print(f"\n[table {i}]")
    print("metadata:", safe_json(getattr(obj, "metadata", {})))
    print(
        "markdown:",
        getattr(obj, "markdown", ""),
        print("context:", getattr(obj, "context", "")),
    )



[table 0]
metadata: {
  "filename": "test.ipynb",
  "origin": "ipynb",
  "pages": [],
  "bboxes": [],
  "cell_index": 1,
  "cell_type": "markdown",
  "output_index": null,
  "mime_type": "text/markdown",
  "sequence_index": 2
}
context: [markdown cell 0]
# Notebook Ingestion Test

This notebook is designed to validate notebook ingestion for text, tables, and images.
[markdown cell 1]
## Markdown Table

| Metric | Value |
|---|---:|
| Precision | 0.91 |
| Recall | 0.87 |
| F1 | 0.89 |

| Metric | Value |
|---|---:|
| Precision | 0.91 |
| Recall | 0.87 |
| F1 | 0.89 |

[markdown cell 2]
## Markdown Attachment Image

Inline attachment below should be extracted as an image.

![inline-chart](attachment:inline_chart.png)
Markdown attachment 'inline_chart.png' in cell 2
[code cell 3]
print("Notebook ingestion smoke test")
[output stream]
Notebook ingestion smoke test\n
[code cell 4]
# Simulated markdown table output
[output text]
table rendered in markdown
| Dataset | Rows | Cols |\n|---|---

In [15]:
# Inspect image objects
for i, obj in enumerate(images[:SHOW]):
    print(f"\n[image {i}]")
    print("metadata:", safe_json(getattr(obj, "metadata", {})))
    print("context:", (getattr(obj, "context", "")))
    print("decoded_image:", safe_json(image_debug_meta(getattr(obj, "base64", ""))))



[image 0]
metadata: {
  "filename": "test.ipynb",
  "origin": "ipynb",
  "pages": [],
  "bboxes": [],
  "cell_index": 2,
  "cell_type": "markdown",
  "output_index": null,
  "mime_type": "image/png",
  "sequence_index": 4
}
context: [markdown cell 0]
# Notebook Ingestion Test

This notebook is designed to validate notebook ingestion for text, tables, and images.
[markdown cell 1]
## Markdown Table

| Metric | Value |
|---|---:|
| Precision | 0.91 |
| Recall | 0.87 |
| F1 | 0.89 |
| Metric | Value |
|---|---:|
| Precision | 0.91 |
| Recall | 0.87 |
| F1 | 0.89 |
[markdown cell 2]
## Markdown Attachment Image

Inline attachment below should be extracted as an image.

![inline-chart](attachment:inline_chart.png)

Markdown attachment 'inline_chart.png' in cell 2

[code cell 3]
print("Notebook ingestion smoke test")
[output stream]
Notebook ingestion smoke test\n
[code cell 4]
# Simulated markdown table output
[output text]
table rendered in markdown
| Dataset | Rows | Cols |\n|---|---:|--

In [17]:
# Optional: run summarization (requires local Ollama model)
RUN_SUMMARIZATION = True
MODEL_NAME = "gemma3:12b"

if RUN_SUMMARIZATION:
    s_texts, s_images, s_tables = summarize_objects(
        texts, images, tables, model_name=MODEL_NAME
    )
    print("summaries generated")
    if s_texts:
        print("text summary sample:", s_texts[0].description)
    if s_tables:
        print("table summary sample:", s_tables[0].description)
    if s_images:
        print("image summary sample:", s_images[0].description)
else:
    print("Skipping summarization. Set RUN_SUMMARIZATION=True to enable.")


  6%|▌         | 1/17 [00:08<02:10,  8.18s/it]


KeyboardInterrupt: 

In [18]:
if s_texts:
    print("text summary sample:", s_texts[0].description)
if s_tables:
    print("table summary sample:", s_tables[0].description)
if s_images:
    print("image summary sample:", s_images[0].description)

text summary sample: This notebook validates the ingestion of notebooks, specifically testing the handling of text, tables, and images.
table summary sample: The table presents performance metrics with Precision at 0.91, Recall at 0.87, and F1 at 0.89. This table is repeated twice.
image summary sample: The image displays a table with metrics like precision, recall, and F1 score presented in two columns. The table is part of a notebook designed to test ingestion of text, tables, and images.
